In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, train_test_split
from sklearn.datasets import make_classification
from sklearn.isotonic import IsotonicRegression

# 1. Setup Ambiguous Data
X, _ = make_classification(n_samples=1000, n_features=10, random_state=42)
y_soft = np.clip(1 / (1 + np.exp(-X[:, 0])) + np.random.normal(0, 0.1, 1000), 0, 1)

# 2. CVAP Loop: Get out-of-sample scores
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = np.zeros(len(X))
for train_idx, calib_idx in kf.split(X):
    clf = RandomForestClassifier(n_estimators=100).fit(X[train_idx], (y_soft[train_idx] > 0.5).astype(int))
    cv_scores[calib_idx] = clf.predict_proba(X[calib_idx])[:, 1]

# 3. Test Set
X_train_final, X_test, y_soft_train, y_soft_test = train_test_split(X, y_soft, test_size=0.2, random_state=42)
final_clf = RandomForestClassifier(n_estimators=100).fit(X_train_final, (y_soft_train > 0.5).astype(int))
p_test = final_clf.predict_proba(X_test)[:, 1]

# 4. Manual Venn-ABERS Logic (The Petej/Vovk Method)
def run_va_research(cal_scores, cal_labels, test_scores):
    p0_list, p1_list = [], []
    
    for p_star in test_scores:
        # Create augmented calibration sets for the test point
        # Assumes test label is 0
        scores_0 = np.append(cal_scores, p_star)
        labels_0 = np.append(cal_labels, 0)
        iso0 = IsotonicRegression(out_of_bounds='clip').fit(scores_0, labels_0)
        p0_list.append(iso0.predict([p_star])[0])
        
        # Assumes test label is 1
        scores_1 = np.append(cal_scores, p_star)
        labels_1 = np.append(cal_labels, 1)
        iso1 = IsotonicRegression(out_of_bounds='clip').fit(scores_1, labels_1)
        p1_list.append(iso1.predict([p_star])[0])
        
    return np.array(p0_list), np.array(p1_list)

# --- THE COMPARISON ---

# Method A: CVAP on Hard Labels
y_cv_hard = (y_soft > 0.5).astype(int)
p0_hard, p1_hard = run_va_research(cv_scores, y_cv_hard, p_test)

# Method B: CVAP on Soft Labels (Averaged Truth Research)
# This works because IsotonicRegression natively accepts continuous y!
p0_soft, p1_soft = run_va_research(cv_scores, y_soft, p_test)

# 5. Metrics
width_hard = p1_hard - p0_hard
width_soft = p1_soft - p0_soft

print(f"CVAP (Hard Labels) Mean Width: {width_hard.mean():.4f}")
print(f"CVAP (Soft Labels) Mean Width: {width_soft.mean():.4f}")

CVAP (Hard Labels) Mean Width: 0.0248
CVAP (Soft Labels) Mean Width: 0.0273


In [4]:
# ============================================================
# Ambiguous-truth Venn-Abers research prototype
# Synthetic binary dataset now; CIFAR-10H / medical hooks below
# ============================================================

from __future__ import annotations

import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, brier_score_loss, log_loss
from sklearn.isotonic import IsotonicRegression

# Assumed exposed in your environment, per your note.
# Interface mirrors ivar_example.ipynb:
#   va = VennAbersRegressor(estimator=model, inductive=False, n_splits=10, random_state=seed)
#   va.fit(X_train, y_train, m=1)
#   preds, aux = va.predict(X_test)
from venn_abers import VennAbersRegressor


# ============================================================
# Utilities
# ============================================================

def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def soft_brier(y_mean_true: np.ndarray, p_hat: np.ndarray) -> float:
    """
    Brier score against the empirical annotator mean.
    Interpretable as squared error for Bernoulli parameter estimation.
    """
    return float(np.mean((y_mean_true - p_hat) ** 2))


def majority_vote_from_mean(y_mean: np.ndarray) -> np.ndarray:
    return (y_mean >= 0.5).astype(int)


def clipped_probs(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return np.clip(p, eps, 1.0 - eps)


# ============================================================
# Synthetic ambiguous-truth data
# ============================================================

@dataclass
class SyntheticAmbiguousDataset:
    X_items: np.ndarray               # shape [n_items, n_features]
    y_mean: np.ndarray               # empirical mean label per item
    y_vote: np.ndarray               # majority vote per item
    n_annotators: np.ndarray         # number of labels per item
    disagreement: np.ndarray         # 2 p (1-p)
    item_ids: np.ndarray             # shape [n_items]
    long_df: pd.DataFrame            # annotator-level dataframe


def make_synthetic_ambiguous_binary_dataset(
    n_items: int = 4000,
    n_features: int = 12,
    min_annotators: int = 5,
    max_annotators: int = 25,
    seed: int = 7,
) -> SyntheticAmbiguousDataset:
    """
    Generates a binary classification dataset with item-dependent annotator ambiguity.

    Construction:
      - latent semantic score s(x) determines the base tendency for class 1
      - ambiguity a(x) reduces confidence toward 0.5 for difficult examples
      - each item has multiple annotator labels sampled independently from Bernoulli(theta_i)

    theta_i is the probability a fresh annotator assigns label 1.
    """
    rng = np.random.default_rng(seed)

    X = rng.normal(size=(n_items, n_features))

    # Latent signal
    linear = (
        1.4 * X[:, 0]
        - 1.1 * X[:, 1]
        + 0.7 * X[:, 2] * X[:, 3]
        - 0.9 * np.sin(X[:, 4])
        + 0.4 * X[:, 5] ** 2
    )

    base_prob = sigmoid(linear)

    # Ambiguity score in [0, 1]:
    # high ambiguity around certain regions of feature space
    amb_raw = (
        1.0
        - np.abs(sigmoid(0.8 * X[:, 6] - 0.6 * X[:, 7] + 0.5 * X[:, 8]) - 0.5) * 2.0
    )
    ambiguity = np.clip(amb_raw, 0.0, 1.0)

    # Pull probabilities toward 0.5 when ambiguity is high
    theta = (1.0 - 0.75 * ambiguity) * base_prob + (0.75 * ambiguity) * 0.5
    theta = np.clip(theta, 1e-4, 1.0 - 1e-4)

    item_ids = np.arange(n_items)
    rows: List[Dict] = []

    for i in range(n_items):
        m_i = int(rng.integers(min_annotators, max_annotators + 1))
        y_ij = rng.binomial(1, theta[i], size=m_i)

        for j in range(m_i):
            row = {
                "item_id": i,
                "annotator_id": j,
                "y": int(y_ij[j]),
            }
            for k in range(n_features):
                row[f"x{k}"] = float(X[i, k])
            rows.append(row)

    long_df = pd.DataFrame(rows)

    agg = (
        long_df.groupby("item_id")["y"]
        .agg(y_mean="mean", n_annotators="size")
        .reset_index()
    )
    agg["y_vote"] = majority_vote_from_mean(agg["y_mean"].to_numpy())
    agg["disagreement"] = 2.0 * agg["y_mean"] * (1.0 - agg["y_mean"])

    item_feature_df = long_df.drop_duplicates("item_id")[["item_id"] + [f"x{k}" for k in range(n_features)]]
    item_df = item_feature_df.merge(agg, on="item_id", how="inner").sort_values("item_id").reset_index(drop=True)

    return SyntheticAmbiguousDataset(
        X_items=item_df[[f"x{k}" for k in range(n_features)]].to_numpy(),
        y_mean=item_df["y_mean"].to_numpy(),
        y_vote=item_df["y_vote"].to_numpy(),
        n_annotators=item_df["n_annotators"].to_numpy(),
        disagreement=item_df["disagreement"].to_numpy(),
        item_ids=item_df["item_id"].to_numpy(),
        long_df=long_df,
    )


# ============================================================
# Item-level / annotator-level views
# ============================================================

def build_long_view(long_df: pd.DataFrame, feature_cols: List[str]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns annotator-level design matrix, labels, and group ids.
    """
    X_long = long_df[feature_cols].to_numpy()
    y_long = long_df["y"].to_numpy().astype(int)
    groups = long_df["item_id"].to_numpy().astype(int)
    return X_long, y_long, groups


def split_by_item(
    item_ids: np.ndarray,
    test_size: float = 0.25,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Group-aware split: entire items stay together.
    """
    dummy_X = np.zeros((len(item_ids), 1))
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(gss.split(dummy_X, groups=item_ids))
    return train_idx, test_idx


# ============================================================
# Models
# ============================================================

class MeanLabelRegressor:
    """
    Average-label approach:
      fit a regressor on item-level y_mean in [0,1],
      then Venn-Abers regression on top.
    """

    def __init__(self, random_state: int = 0, inductive: bool = False, n_splits: int = 10, m: int = 1):
        self.base = HistGradientBoostingRegressor(
            max_depth=5,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.va = VennAbersRegressor(
            estimator=self.base,
            inductive=inductive,
            n_splits=n_splits,
            random_state=random_state,
        )
        self.m = m
        self.fitted_ = False

    def fit(self, X: np.ndarray, y_mean: np.ndarray) -> "MeanLabelRegressor":
        self.va.fit(X, y_mean, m=self.m)
        self.fitted_ = True
        return self

    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        """
        Assumes predict returns (predictions, aux), like ivar_example.
        We interpret aux as intervals or extra uncertainty information if available.
        """
        pred, aux = self.va.predict(X)
        pred = np.asarray(pred, dtype=float)
        pred = np.clip(pred, 0.0, 1.0)
        return pred, aux


class AnnotatorLevelClassifier:
    """
    Individual-annotator baseline:
      fit a classifier on repeated rows (same item, many annotators).
    """

    def __init__(self, random_state: int = 0):
        self.model = HistGradientBoostingClassifier(
            max_depth=5,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.fitted_ = False

    def fit(self, X_long: np.ndarray, y_long: np.ndarray) -> "AnnotatorLevelClassifier":
        self.model.fit(X_long, y_long)
        self.fitted_ = True
        return self

    def predict_item_probability(self, X_items: np.ndarray) -> np.ndarray:
        p = self.model.predict_proba(X_items)[:, 1]
        return clipped_probs(p)


class VoteLabelClassifier:
    """
    Majority-vote baseline.
    """

    def __init__(self, random_state: int = 0):
        self.model = HistGradientBoostingClassifier(
            max_depth=5,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.fitted_ = False

    def fit(self, X: np.ndarray, y_vote: np.ndarray) -> "VoteLabelClassifier":
        self.model.fit(X, y_vote)
        self.fitted_ = True
        return self

    def predict_item_probability(self, X_items: np.ndarray) -> np.ndarray:
        p = self.model.predict_proba(X_items)[:, 1]
        return clipped_probs(p)


# ============================================================
# Optional: simple interval extraction helper
# ============================================================

def parse_va_aux_to_interval(
    pred: np.ndarray,
    aux: Optional[np.ndarray],
    fallback_half_width: float = 0.05,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Because different builds may return different 'aux' formats, this helper
    tries to extract [lower, upper] robustly.

    Expected cases:
      1) aux is shape [n, 2] => [lower, upper]
      2) aux is None => create a narrow fallback interval around pred
      3) anything else => fallback interval around pred

    Replace this function with the exact library-specific parsing once you inspect
    your local installed return type.
    """
    pred = np.asarray(pred, dtype=float)

    if aux is not None:
        aux_arr = np.asarray(aux)
        if aux_arr.ndim == 2 and aux_arr.shape[1] == 2:
            lower = np.clip(aux_arr[:, 0], 0.0, 1.0)
            upper = np.clip(aux_arr[:, 1], 0.0, 1.0)
            lower, upper = np.minimum(lower, upper), np.maximum(lower, upper)
            return lower, upper

    lower = np.clip(pred - fallback_half_width, 0.0, 1.0)
    upper = np.clip(pred + fallback_half_width, 0.0, 1.0)
    return lower, upper


# ============================================================
# Evaluation
# ============================================================

def evaluate_against_test_items(
    y_mean_true: np.ndarray,
    y_vote_true: np.ndarray,
    p_hat: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
    disagreement_true: np.ndarray,
) -> Dict[str, float]:
    width = upper - lower

    metrics = {
        "rmse_to_y_mean": rmse(y_mean_true, p_hat),
        "soft_brier_to_y_mean": soft_brier(y_mean_true, p_hat),
        "vote_brier": float(np.mean((y_vote_true - p_hat) ** 2)),
        "avg_interval_width": float(np.mean(width)),
        "corr_width_vs_disagreement": float(np.corrcoef(width, disagreement_true)[0, 1]),
    }
    return metrics


def fresh_annotator_brier(
    p_hat: np.ndarray,
    y_mean_true: np.ndarray,
    n_draws: int = 200,
    seed: int = 0,
) -> float:
    """
    Monte Carlo estimate:
      draw fresh annotator labels from Bernoulli(y_mean_true)
      evaluate Brier score of p_hat
    """
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_draws):
        y_new = rng.binomial(1, y_mean_true)
        scores.append(brier_score_loss(y_new, p_hat))
    return float(np.mean(scores))


# ============================================================
# Monte Carlo conformal under ambiguous truth (binary version)
# ============================================================

def nonconformity_binary(prob_pos: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Split-conformal nonconformity score:
      smaller predicted probability for the realized label => larger nonconformity.
    """
    p = clipped_probs(prob_pos)
    return np.where(y == 1, 1.0 - p, p)


def conformal_threshold(scores: np.ndarray, alpha: float) -> float:
    """
    Standard split conformal quantile:
      q = ceil((n+1)(1-alpha))/n style via empirical quantile on nonconformity.
    """
    n = len(scores)
    k = int(np.ceil((n + 1) * (1.0 - alpha))) - 1
    k = np.clip(k, 0, n - 1)
    return float(np.sort(scores)[k])


def monte_carlo_ambiguous_conformal_binary(
    p_cal: np.ndarray,
    y_mean_cal: np.ndarray,
    p_test: np.ndarray,
    alpha: float = 0.1,
    n_mc: int = 200,
    use_interval: bool = False,
    lower_cal: Optional[np.ndarray] = None,
    upper_cal: Optional[np.ndarray] = None,
    lower_test: Optional[np.ndarray] = None,
    upper_test: Optional[np.ndarray] = None,
    seed: int = 0,
) -> Dict[str, np.ndarray]:
    """
    Binary analogue of Monte Carlo CP under ambiguous truth.

    If use_interval=False:
      pseudo-labels are drawn from Bernoulli(y_mean_cal)

    If use_interval=True:
      theta_i is first drawn from Uniform([lower_i, upper_i]), then label from Bernoulli(theta_i)

    Prediction set for test x is subset of {0,1}.
    """
    rng = np.random.default_rng(seed)

    # Precompute class-conditional nonconformity for test points
    p_test = clipped_probs(p_test)
    nc_test_if_0 = p_test               # score if label were 0
    nc_test_if_1 = 1.0 - p_test         # score if label were 1

    include_0_counts = np.zeros_like(p_test, dtype=float)
    include_1_counts = np.zeros_like(p_test, dtype=float)

    for _ in range(n_mc):
        if use_interval:
            if lower_cal is None or upper_cal is None:
                raise ValueError("Interval calibration requested but lower_cal/upper_cal missing.")
            theta_cal = rng.uniform(lower_cal, upper_cal)
        else:
            theta_cal = y_mean_cal

        y_pseudo = rng.binomial(1, clipped_probs(theta_cal))
        cal_scores = nonconformity_binary(p_cal, y_pseudo)
        q_hat = conformal_threshold(cal_scores, alpha=alpha)

        include_0_counts += (nc_test_if_0 <= q_hat).astype(float)
        include_1_counts += (nc_test_if_1 <= q_hat).astype(float)

    include_0 = include_0_counts / n_mc >= 0.5
    include_1 = include_1_counts / n_mc >= 0.5

    return {
        "include_0_prob": include_0_counts / n_mc,
        "include_1_prob": include_1_counts / n_mc,
        "include_0": include_0,
        "include_1": include_1,
        "set_size": include_0.astype(int) + include_1.astype(int),
    }


# ============================================================
# Main experiment
# ============================================================

def run_synthetic_experiment(seed: int = 0) -> None:
    ds = make_synthetic_ambiguous_binary_dataset(
        n_items=4000,
        n_features=12,
        min_annotators=5,
        max_annotators=25,
        seed=seed,
    )

    feature_cols = [c for c in ds.long_df.columns if c.startswith("x")]
    train_idx, test_idx = split_by_item(ds.item_ids, test_size=0.25, seed=seed)

    X_train = ds.X_items[train_idx]
    X_test = ds.X_items[test_idx]

    y_mean_train = ds.y_mean[train_idx]
    y_mean_test = ds.y_mean[test_idx]

    y_vote_train = ds.y_vote[train_idx]
    y_vote_test = ds.y_vote[test_idx]

    disagreement_test = ds.disagreement[test_idx]
    item_ids_train = ds.item_ids[train_idx]
    item_ids_test = ds.item_ids[test_idx]

    # Long-form annotator labels restricted to train items
    train_item_set = set(item_ids_train.tolist())
    long_train = ds.long_df[ds.long_df["item_id"].isin(train_item_set)].copy()

    X_long_train, y_long_train, groups_long_train = build_long_view(long_train, feature_cols)

    # --------------------------------------------------------
    # 1) Average-label Venn-Abers regression
    # --------------------------------------------------------
    avg_va = MeanLabelRegressor(random_state=seed, inductive=False, n_splits=10, m=1)
    avg_va.fit(X_train, y_mean_train)
    p_avg, aux_avg = avg_va.predict(X_test)
    lower_avg, upper_avg = parse_va_aux_to_interval(p_avg, aux_avg, fallback_half_width=0.08)

    avg_metrics = evaluate_against_test_items(
        y_mean_true=y_mean_test,
        y_vote_true=y_vote_test,
        p_hat=p_avg,
        lower=lower_avg,
        upper=upper_avg,
        disagreement_true=disagreement_test,
    )
    avg_metrics["fresh_annotator_brier"] = fresh_annotator_brier(p_avg, y_mean_test, seed=seed)

    # --------------------------------------------------------
    # 2) Annotator-level classifier baseline
    # --------------------------------------------------------
    ann_clf = AnnotatorLevelClassifier(random_state=seed)
    ann_clf.fit(X_long_train, y_long_train)
    p_ann = ann_clf.predict_item_probability(X_test)

    # no native interval here; use simple symmetric placeholder for comparability
    lower_ann = np.clip(p_ann - 0.05, 0.0, 1.0)
    upper_ann = np.clip(p_ann + 0.05, 0.0, 1.0)

    ann_metrics = evaluate_against_test_items(
        y_mean_true=y_mean_test,
        y_vote_true=y_vote_test,
        p_hat=p_ann,
        lower=lower_ann,
        upper=upper_ann,
        disagreement_true=disagreement_test,
    )
    ann_metrics["fresh_annotator_brier"] = fresh_annotator_brier(p_ann, y_mean_test, seed=seed)

    # --------------------------------------------------------
    # 3) Vote-label classifier baseline
    # --------------------------------------------------------
    vote_clf = VoteLabelClassifier(random_state=seed)
    vote_clf.fit(X_train, y_vote_train)
    p_vote = vote_clf.predict_item_probability(X_test)

    lower_vote = np.clip(p_vote - 0.05, 0.0, 1.0)
    upper_vote = np.clip(p_vote + 0.05, 0.0, 1.0)

    vote_metrics = evaluate_against_test_items(
        y_mean_true=y_mean_test,
        y_vote_true=y_vote_test,
        p_hat=p_vote,
        lower=lower_vote,
        upper=upper_vote,
        disagreement_true=disagreement_test,
    )
    vote_metrics["fresh_annotator_brier"] = fresh_annotator_brier(p_vote, y_mean_test, seed=seed)

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------
    results = pd.DataFrame(
        {
            "avg_label_va_regression": avg_metrics,
            "annotator_level_classifier": ann_metrics,
            "vote_label_classifier": vote_metrics,
        }
    ).T

    print("\n=== Item-level evaluation ===")
    print(results.round(4))

    # --------------------------------------------------------
    # Optional: binary Monte Carlo conformal on held-out calibration split
    # --------------------------------------------------------
    # Create inner train/cal split from the original training items
    inner_train_idx, cal_idx = split_by_item(item_ids_train, test_size=0.35, seed=seed + 1)

    X_inner_train = X_train[inner_train_idx]
    X_cal = X_train[cal_idx]

    y_mean_inner_train = y_mean_train[inner_train_idx]
    y_mean_cal = y_mean_train[cal_idx]

    # Refit average-label VA on inner-train, get probs on cal/test
    avg_va_cp = MeanLabelRegressor(random_state=seed + 1, inductive=False, n_splits=10, m=1)
    avg_va_cp.fit(X_inner_train, y_mean_inner_train)

    p_cal, aux_cal = avg_va_cp.predict(X_cal)
    p_test_cp, aux_test_cp = avg_va_cp.predict(X_test)

    lower_cal, upper_cal = parse_va_aux_to_interval(p_cal, aux_cal, fallback_half_width=0.08)
    lower_test, upper_test = parse_va_aux_to_interval(p_test_cp, aux_test_cp, fallback_half_width=0.08)

    cp_plain = monte_carlo_ambiguous_conformal_binary(
        p_cal=p_cal,
        y_mean_cal=y_mean_cal,
        p_test=p_test_cp,
        alpha=0.1,
        n_mc=300,
        use_interval=False,
        seed=seed,
    )

    cp_interval = monte_carlo_ambiguous_conformal_binary(
        p_cal=p_cal,
        y_mean_cal=y_mean_cal,
        p_test=p_test_cp,
        alpha=0.1,
        n_mc=300,
        use_interval=True,
        lower_cal=lower_cal,
        upper_cal=upper_cal,
        lower_test=lower_test,
        upper_test=upper_test,
        seed=seed,
    )

    # Evaluate aggregated coverage on test items using empirical y_mean_test
    # For binary sets, aggregated coverage for item i is:
    #   y_mean_i * I(1 in C_i) + (1-y_mean_i) * I(0 in C_i)
    def aggregated_coverage(y_mean: np.ndarray, include_0: np.ndarray, include_1: np.ndarray) -> float:
        return float(np.mean(y_mean * include_1.astype(float) + (1.0 - y_mean) * include_0.astype(float)))

    print("\n=== Monte Carlo CP (binary ambiguous-truth analogue) ===")
    print({
        "plain_mc_cp_aggregated_coverage": round(
            aggregated_coverage(y_mean_test, cp_plain["include_0"], cp_plain["include_1"]), 4
        ),
        "plain_mc_cp_avg_set_size": round(float(np.mean(cp_plain["set_size"])), 4),
        "interval_mc_cp_aggregated_coverage": round(
            aggregated_coverage(y_mean_test, cp_interval["include_0"], cp_interval["include_1"]), 4
        ),
        "interval_mc_cp_avg_set_size": round(float(np.mean(cp_interval["set_size"])), 4),
    })





In [6]:
run_synthetic_experiment(seed=1)

/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:



=== Item-level evaluation ===
                            rmse_to_y_mean  soft_brier_to_y_mean  vote_brier  \
avg_label_va_regression             0.1537                0.0236      0.1748   
annotator_level_classifier          0.1555                0.0242      0.1716   
vote_label_classifier               0.2439                0.0595      0.1440   

                            avg_interval_width  corr_width_vs_disagreement  \
avg_label_va_regression                 0.0564                     -0.1155   
annotator_level_classifier              0.1000                     -0.0763   
vote_label_classifier                   0.0954                      0.1830   

                            fresh_annotator_brier  
avg_label_va_regression                    0.2237  
annotator_level_classifier                 0.2241  
vote_label_classifier                      0.2585  

=== Monte Carlo CP (binary ambiguous-truth analogue) ===
{'plain_mc_cp_aggregated_coverage': 0.909, 'plain_mc_cp_avg_set_size'

/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


In [8]:
from __future__ import annotations

import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Assumed available in your environment
from venn_abers import VennAbersRegressor


# ============================================================
# Utilities
# ============================================================

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def soft_brier(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true - y_pred) ** 2))


def clip01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return np.clip(x, eps, 1.0 - eps)


def normalize_rows(p: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    p = np.maximum(p, eps)
    return p / p.sum(axis=1, keepdims=True)


def argmax_labels(soft_labels: np.ndarray) -> np.ndarray:
    return np.argmax(soft_labels, axis=1).astype(int)


def entropy_rows(p: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log(p), axis=1)


# ============================================================
# Data container
# ============================================================

@dataclass
class Cifar10HAmbiguousData:
    X: np.ndarray                     # [n_items, d] embeddings or logits
    soft_labels: np.ndarray           # [n_items, 10], empirical human label distribution
    hard_labels: np.ndarray           # [n_items], e.g. argmax of soft labels
    human_labels: Optional[np.ndarray] = None   # [n_items, n_ann] integer labels if available


# ============================================================
# Preparation helpers
# ============================================================

def build_one_vs_rest_targets(
    soft_labels: np.ndarray,
    class_idx: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      y_mean_k: empirical probability human assigns class k
      y_hard_k: 1 if argmax class is k, else 0
    """
    y_mean_k = soft_labels[:, class_idx].astype(float)
    y_hard_k = (np.argmax(soft_labels, axis=1) == class_idx).astype(int)
    return y_mean_k, y_hard_k


def build_annotator_level_binary(
    X: np.ndarray,
    human_labels: np.ndarray,
    class_idx: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Expands image-level embeddings and human labels into annotator-level rows.

    Returns:
      X_long: [n_items*n_ann, d]
      y_long: binary labels for class_idx
      item_ids: item id per long row
    """
    n_items, d = X.shape
    n_ann = human_labels.shape[1]

    X_long = np.repeat(X, repeats=n_ann, axis=0)
    y_long = (human_labels.reshape(-1) == class_idx).astype(int)
    item_ids = np.repeat(np.arange(n_items), repeats=n_ann)
    return X_long, y_long, item_ids


def train_test_split_items(
    X: np.ndarray,
    soft_labels: np.ndarray,
    test_size: float = 0.3,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Stratify by argmax class to keep the split balanced.
    """
    y_strat = np.argmax(soft_labels, axis=1)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y_strat))
    return train_idx, test_idx


# ============================================================
# Models
# ============================================================

class AvgLabelVARegressor:
    """
    Average-label Venn-Abers regression for one-vs-rest class probability.
    """

    def __init__(self, random_state: int = 0, inductive: bool = False, n_splits: int = 10, m: int = 1):
        base = HistGradientBoostingRegressor(
            max_depth=5,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.model = VennAbersRegressor(
            estimator=base,
            inductive=inductive,
            n_splits=n_splits,
            random_state=random_state,
        )
        self.m = m

    def fit(self, X: np.ndarray, y_mean: np.ndarray) -> "AvgLabelVARegressor":
        self.model.fit(X, y_mean, m=self.m)
        return self

    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        pred, aux = self.model.predict(X)
        pred = np.clip(np.asarray(pred, dtype=float), 0.0, 1.0)
        return pred, aux


class AnnotatorLevelBinaryClassifier:
    """
    One-vs-rest classifier trained on individual human labels.
    """

    def __init__(self, random_state: int = 0):
        self.model = HistGradientBoostingClassifier(
            max_depth=5,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            random_state=random_state,
        )

    def fit(self, X_long: np.ndarray, y_long: np.ndarray) -> "AnnotatorLevelBinaryClassifier":
        self.model.fit(X_long, y_long)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return clip01(self.model.predict_proba(X)[:, 1])


class HardLabelBinaryClassifier:
    """
    One-vs-rest classifier trained on argmax human class.
    """

    def __init__(self, random_state: int = 0):
        self.model = HistGradientBoostingClassifier(
            max_depth=5,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            random_state=random_state,
        )

    def fit(self, X: np.ndarray, y_hard: np.ndarray) -> "HardLabelBinaryClassifier":
        self.model.fit(X, y_hard)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return clip01(self.model.predict_proba(X)[:, 1])


# ============================================================
# Interval parsing
# ============================================================

def parse_va_interval(
    pred: np.ndarray,
    aux: Optional[np.ndarray],
    fallback_half_width: float = 0.05,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Replace this with exact extraction once you inspect your installed package.
    """
    pred = np.asarray(pred, dtype=float)

    if aux is not None:
        aux_arr = np.asarray(aux)
        if aux_arr.ndim == 2 and aux_arr.shape[1] == 2:
            lower = np.clip(aux_arr[:, 0], 0.0, 1.0)
            upper = np.clip(aux_arr[:, 1], 0.0, 1.0)
            return np.minimum(lower, upper), np.maximum(lower, upper)

    lower = np.clip(pred - fallback_half_width, 0.0, 1.0)
    upper = np.clip(pred + fallback_half_width, 0.0, 1.0)
    return lower, upper


# ============================================================
# Classwise evaluation
# ============================================================

def evaluate_binary_classwise(
    y_mean_true: np.ndarray,
    y_hard_true: np.ndarray,
    p_hat: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
) -> Dict[str, float]:
    width = upper - lower
    disagreement = 2.0 * y_mean_true * (1.0 - y_mean_true)

    return {
        "rmse_to_y_mean": rmse(y_mean_true, p_hat),
        "soft_brier_to_y_mean": soft_brier(y_mean_true, p_hat),
        "hard_brier": float(np.mean((y_hard_true - p_hat) ** 2)),
        "avg_interval_width": float(np.mean(width)),
        "corr_width_vs_disagreement": float(np.corrcoef(width, disagreement)[0, 1]),
    }


def fresh_annotator_brier_multiclass_from_binary(
    p_hat_binary: np.ndarray,
    sampled_binary_labels: np.ndarray,
) -> float:
    return float(np.mean((sampled_binary_labels - p_hat_binary) ** 2))


# ============================================================
# Monte Carlo CP, binary one-vs-rest
# ============================================================

def nonconformity_binary(prob_pos: np.ndarray, y: np.ndarray) -> np.ndarray:
    p = clip01(prob_pos)
    return np.where(y == 1, 1.0 - p, p)


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    k = int(np.ceil((n + 1) * (1.0 - alpha))) - 1
    k = int(np.clip(k, 0, n - 1))
    return float(np.sort(scores)[k])


def ambiguous_mc_cp_binary(
    p_cal: np.ndarray,
    y_mean_cal: np.ndarray,
    p_test: np.ndarray,
    alpha: float = 0.1,
    n_mc: int = 200,
    lower_cal: Optional[np.ndarray] = None,
    upper_cal: Optional[np.ndarray] = None,
    use_interval: bool = False,
    seed: int = 0,
) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)

    p_test = clip01(p_test)
    nc_if_0 = p_test
    nc_if_1 = 1.0 - p_test

    include_0_count = np.zeros_like(p_test, dtype=float)
    include_1_count = np.zeros_like(p_test, dtype=float)

    for _ in range(n_mc):
        if use_interval:
            if lower_cal is None or upper_cal is None:
                raise ValueError("lower_cal and upper_cal are required when use_interval=True")
            theta_cal = rng.uniform(lower_cal, upper_cal)
            theta_cal = clip01(theta_cal)
        else:
            theta_cal = clip01(y_mean_cal)

        y_pseudo = rng.binomial(1, theta_cal)
        scores = nonconformity_binary(p_cal, y_pseudo)
        qhat = conformal_quantile(scores, alpha=alpha)

        include_0_count += (nc_if_0 <= qhat).astype(float)
        include_1_count += (nc_if_1 <= qhat).astype(float)

    include_0_prob = include_0_count / n_mc
    include_1_prob = include_1_count / n_mc

    return {
        "include_0_prob": include_0_prob,
        "include_1_prob": include_1_prob,
        "include_0": include_0_prob >= 0.5,
        "include_1": include_1_prob >= 0.5,
        "set_size": (include_0_prob >= 0.5).astype(int) + (include_1_prob >= 0.5).astype(int),
    }


# ============================================================
# Main CIFAR-10H experiment
# ============================================================

def run_cifar10h_experiment(
    data: Cifar10HAmbiguousData,
    seed: int = 0,
    test_size: float = 0.3,
    alpha: float = 0.1,
    n_mc: int = 300,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      classwise_results: per-class metrics
      summary_results: macro averages across classes
    """
    X = data.X
    soft_labels = normalize_rows(data.soft_labels)
    hard_labels = argmax_labels(soft_labels)

    train_idx, test_idx = train_test_split_items(X, soft_labels, test_size=test_size, seed=seed)

    X_train, X_test = X[train_idx], X[test_idx]
    soft_train, soft_test = soft_labels[train_idx], soft_labels[test_idx]
    hard_train, hard_test = hard_labels[train_idx], hard_labels[test_idx]

    results_rows = []

    for k in range(10):
        y_mean_train_k, y_hard_train_k = build_one_vs_rest_targets(soft_train, k)
        y_mean_test_k, y_hard_test_k = build_one_vs_rest_targets(soft_test, k)

        # -------------------------
        # Average-label VA regression
        # -------------------------
        avg_va = AvgLabelVARegressor(random_state=seed + k, inductive=False, n_splits=10, m=1)
        avg_va.fit(X_train, y_mean_train_k)
        p_avg_k, aux_avg_k = avg_va.predict(X_test)
        lower_avg_k, upper_avg_k = parse_va_interval(p_avg_k, aux_avg_k, fallback_half_width=0.05)

        avg_metrics = evaluate_binary_classwise(
            y_mean_true=y_mean_test_k,
            y_hard_true=y_hard_test_k,
            p_hat=p_avg_k,
            lower=lower_avg_k,
            upper=upper_avg_k,
        )

        # Fresh annotator Brier for this class
        if data.human_labels is not None:
            human_test = data.human_labels[test_idx]
            sampled_binary = (human_test[:, 0] == k).astype(int)  # one deterministic annotator column
            avg_metrics["fresh_annotator_brier"] = fresh_annotator_brier_multiclass_from_binary(p_avg_k, sampled_binary)
        else:
            # Monte Carlo proxy from empirical class probability
            rng = np.random.default_rng(seed + 1000 + k)
            sampled_binary = rng.binomial(1, y_mean_test_k)
            avg_metrics["fresh_annotator_brier"] = fresh_annotator_brier_multiclass_from_binary(p_avg_k, sampled_binary)

        # -------------------------
        # Annotator-level baseline
        # -------------------------
        if data.human_labels is not None:
            X_long_train, y_long_train_k, _ = build_annotator_level_binary(X_train, data.human_labels[train_idx], k)
            ann_clf = AnnotatorLevelBinaryClassifier(random_state=seed + k)
            ann_clf.fit(X_long_train, y_long_train_k)
            p_ann_k = ann_clf.predict(X_test)

            ann_metrics = evaluate_binary_classwise(
                y_mean_true=y_mean_test_k,
                y_hard_true=y_hard_test_k,
                p_hat=p_ann_k,
                lower=np.clip(p_ann_k - 0.05, 0.0, 1.0),
                upper=np.clip(p_ann_k + 0.05, 0.0, 1.0),
            )
            ann_metrics["fresh_annotator_brier"] = fresh_annotator_brier_multiclass_from_binary(p_ann_k, sampled_binary)
        else:
            ann_metrics = None

        # -------------------------
        # Hard-label baseline
        # -------------------------
        hard_clf = HardLabelBinaryClassifier(random_state=seed + k)
        hard_clf.fit(X_train, y_hard_train_k)
        p_hard_k = hard_clf.predict(X_test)

        hard_metrics = evaluate_binary_classwise(
            y_mean_true=y_mean_test_k,
            y_hard_true=y_hard_test_k,
            p_hat=p_hard_k,
            lower=np.clip(p_hard_k - 0.05, 0.0, 1.0),
            upper=np.clip(p_hard_k + 0.05, 0.0, 1.0),
        )
        hard_metrics["fresh_annotator_brier"] = fresh_annotator_brier_multiclass_from_binary(p_hard_k, sampled_binary)

        # -------------------------
        # Optional MC conformal for avg-label VA
        # inner split from train into proper train / calibration
        # -------------------------
        inner_split = StratifiedShuffleSplit(n_splits=1, test_size=0.35, random_state=seed + 300 + k)
        inner_train_idx_rel, cal_idx_rel = next(inner_split.split(X_train, y_hard_train_k))

        X_inner_train = X_train[inner_train_idx_rel]
        X_cal = X_train[cal_idx_rel]
        y_mean_inner_train_k = y_mean_train_k[inner_train_idx_rel]
        y_mean_cal_k = y_mean_train_k[cal_idx_rel]

        avg_va_cp = AvgLabelVARegressor(random_state=seed + 500 + k, inductive=False, n_splits=10, m=1)
        avg_va_cp.fit(X_inner_train, y_mean_inner_train_k)

        p_cal_k, aux_cal_k = avg_va_cp.predict(X_cal)
        lower_cal_k, upper_cal_k = parse_va_interval(p_cal_k, aux_cal_k, fallback_half_width=0.05)

        cp_plain = ambiguous_mc_cp_binary(
            p_cal=p_cal_k,
            y_mean_cal=y_mean_cal_k,
            p_test=p_avg_k,
            alpha=alpha,
            n_mc=n_mc,
            use_interval=False,
            seed=seed + 700 + k,
        )

        cp_interval = ambiguous_mc_cp_binary(
            p_cal=p_cal_k,
            y_mean_cal=y_mean_cal_k,
            p_test=p_avg_k,
            alpha=alpha,
            n_mc=n_mc,
            lower_cal=lower_cal_k,
            upper_cal=upper_cal_k,
            use_interval=True,
            seed=seed + 800 + k,
        )

        # Aggregated coverage for binary one-vs-rest:
        # E[ 1[y in set] ] under Bernoulli(y_mean_test_k)
        def aggregated_coverage(y_mean: np.ndarray, include_0: np.ndarray, include_1: np.ndarray) -> float:
            return float(np.mean(y_mean * include_1.astype(float) + (1.0 - y_mean) * include_0.astype(float)))

        cp_plain_cov = aggregated_coverage(y_mean_test_k, cp_plain["include_0"], cp_plain["include_1"])
        cp_interval_cov = aggregated_coverage(y_mean_test_k, cp_interval["include_0"], cp_interval["include_1"])

        row_avg = {"class_idx": k, "method": "avg_label_va_regression", **avg_metrics,
                   "mc_cp_cov_plain": cp_plain_cov,
                   "mc_cp_size_plain": float(np.mean(cp_plain["set_size"])),
                   "mc_cp_cov_interval": cp_interval_cov,
                   "mc_cp_size_interval": float(np.mean(cp_interval["set_size"]))}
        results_rows.append(row_avg)

        if ann_metrics is not None:
            row_ann = {"class_idx": k, "method": "annotator_level_classifier", **ann_metrics}
            results_rows.append(row_ann)

        row_hard = {"class_idx": k, "method": "hard_label_classifier", **hard_metrics}
        results_rows.append(row_hard)

    classwise_results = pd.DataFrame(results_rows)

    summary_results = (
        classwise_results
        .groupby("method", as_index=False)
        .mean(numeric_only=True)
        .sort_values("method")
        .reset_index(drop=True)
    )

    return classwise_results, summary_results


# ============================================================
# Multiclass aggregation from classwise predictors
# ============================================================

def fit_multiclass_avg_label_va(
    X_train: np.ndarray,
    soft_train: np.ndarray,
    X_test: np.ndarray,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fits one-vs-rest AvgLabelVARegressor for all 10 classes.
    Returns:
      P_pred: [n_test, 10] classwise predicted probabilities, row-normalized
      L: [n_test, 10] lower bounds, row-wise not normalized
      U: [n_test, 10] upper bounds, row-wise not normalized
    """
    preds = []
    lowers = []
    uppers = []

    for k in range(10):
        model = AvgLabelVARegressor(random_state=seed + k, inductive=False, n_splits=10, m=1)
        model.fit(X_train, soft_train[:, k])
        p_k, aux_k = model.predict(X_test)
        l_k, u_k = parse_va_interval(p_k, aux_k, fallback_half_width=0.05)
        preds.append(p_k)
        lowers.append(l_k)
        uppers.append(u_k)

    P = np.stack(preds, axis=1)
    L = np.stack(lowers, axis=1)
    U = np.stack(uppers, axis=1)

    P = normalize_rows(P)
    return P, L, U


def multiclass_metrics(
    soft_true: np.ndarray,
    P_pred: np.ndarray,
    hard_true: np.ndarray,
) -> Dict[str, float]:
    """
    Simple multiclass metrics against ambiguous truth.
    """
    P_pred = normalize_rows(P_pred)
    soft_true = normalize_rows(soft_true)

    rmse_prob = float(np.sqrt(np.mean((soft_true - P_pred) ** 2)))
    soft_brier_mc = float(np.mean(np.sum((soft_true - P_pred) ** 2, axis=1)))
    hard_acc = float(np.mean(np.argmax(P_pred, axis=1) == hard_true))
    pred_entropy = float(np.mean(entropy_rows(P_pred)))
    true_entropy = float(np.mean(entropy_rows(soft_true)))

    return {
        "rmse_prob_matrix": rmse_prob,
        "soft_brier_multiclass": soft_brier_mc,
        "hard_accuracy_vs_argmax": hard_acc,
        "avg_pred_entropy": pred_entropy,
        "avg_true_entropy": true_entropy,
    }


# ============================================================
# Example entry point
# ============================================================

def main():
    # --------------------------------------------------------
    # Replace this block with your real loading code.
    #
    # Expected:
    #   X: [n_items, d] embeddings or logits
    #   soft_labels: [n_items, 10], human empirical label distribution
    #   human_labels: optional [n_items, n_ann]
    # --------------------------------------------------------
    rng = np.random.default_rng(0)
    n_items = 10000
    d = 128

    X = rng.normal(size=(n_items, d))

    # Fake CIFAR-10H-like soft labels just to make the script runnable
    logits = rng.normal(size=(n_items, 10))
    soft_labels = np.exp(logits)
    soft_labels = soft_labels / soft_labels.sum(axis=1, keepdims=True)

    # Optional fake annotator labels: 50 labels per item
    n_ann = 50
    human_labels = np.array(
        [rng.choice(10, size=n_ann, p=soft_labels[i]) for i in range(n_items)],
        dtype=int
    )

    data = Cifar10HAmbiguousData(
        X=X,
        soft_labels=soft_labels,
        hard_labels=np.argmax(soft_labels, axis=1),
        human_labels=human_labels,
    )

    classwise_results, summary_results = run_cifar10h_experiment(
        data=data,
        seed=42,
        test_size=0.3,
        alpha=0.1,
        n_mc=200,
    )

    print("\n=== CIFAR-10H classwise results ===")
    print(classwise_results.head(20).round(4))

    print("\n=== CIFAR-10H macro summary ===")
    print(summary_results.round(4))

    # Optional multiclass aggregation using the avg-label VA models
    train_idx, test_idx = train_test_split_items(data.X, data.soft_labels, test_size=0.3, seed=42)
    P_pred, L, U = fit_multiclass_avg_label_va(
        X_train=data.X[train_idx],
        soft_train=data.soft_labels[train_idx],
        X_test=data.X[test_idx],
        seed=123,
    )

    multi_metrics = multiclass_metrics(
        soft_true=data.soft_labels[test_idx],
        P_pred=P_pred,
        hard_true=np.argmax(data.soft_labels[test_idx], axis=1),
    )

    print("\n=== Multiclass avg-label VA summary ===")
    print({k: round(v, 4) for k, v in multi_metrics.items()})




In [9]:
main()

/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpete


=== CIFAR-10H classwise results ===
    class_idx                      method  rmse_to_y_mean  \
0           0     avg_label_va_regression          0.0984   
1           0  annotator_level_classifier          0.0994   
2           0       hard_label_classifier          0.1102   
3           1     avg_label_va_regression          0.0970   
4           1  annotator_level_classifier          0.0983   
5           1       hard_label_classifier          0.1101   
6           2     avg_label_va_regression          0.0995   
7           2  annotator_level_classifier          0.1010   
8           2       hard_label_classifier          0.1121   
9           3     avg_label_va_regression          0.0981   
10          3  annotator_level_classifier          0.0997   
11          3       hard_label_classifier          0.1116   
12          4     avg_label_va_regression          0.0982   
13          4  annotator_level_classifier          0.0996   
14          4       hard_label_classifier       

/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/ivanpete


=== Multiclass avg-label VA summary ===
{'rmse_prob_matrix': 0.0972, 'soft_brier_multiclass': 0.0944, 'hard_accuracy_vs_argmax': 0.0987, 'avg_pred_entropy': 2.2972, 'avg_true_entropy': 1.9262}


/Users/ivanpetej/miniconda/envs/venn_abers_github/lib/python3.11/site-packages/venn_abers/venn_abers.py:177: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


In [20]:
from __future__ import annotations

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.neighbors import KernelDensity

# Assumed available in your environment
from venn_abers import VennAbersCalibrator


# ============================================================
# Global config
# ============================================================

RANDOM_SEED = 13
OUTPUT_DIR = "va_calibration_imprecision_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# Utilities
# ============================================================

def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def clip01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return np.clip(x, eps, 1.0 - eps)


def true_probability_1d(x: np.ndarray) -> np.ndarray:
    """
    Smooth monotone truth used in Experiments A/B/C.
    """
    return 0.1 + 0.8 * sigmoid(6.0 * x)


def ambiguity_from_p(p: np.ndarray) -> np.ndarray:
    return 2.0 * p * (1.0 - p)


def sample_labels_from_p(p: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return rng.binomial(1, clip01(p)).astype(int)


def corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def ensure_2d_column(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    if x.ndim == 1:
        return x.reshape(-1, 1)
    return x


def save_plot(fig: plt.Figure, filename: str) -> None:
    path = os.path.join(OUTPUT_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Data generators
# ============================================================

def sample_x_dense_uniform(n: int, rng: np.random.Generator) -> np.ndarray:
    return rng.uniform(-2.0, 2.0, size=n)


def sample_x_center_dense_tail_sparse(n: int, rng: np.random.Generator) -> np.ndarray:
    """
    Dense near 0, sparse in tails.
    """
    mix = rng.uniform(size=n)
    x = np.empty(n)

    center_mask = mix < 0.8
    n_center = center_mask.sum()
    n_tail = n - n_center

    x[center_mask] = rng.normal(loc=0.0, scale=0.35, size=n_center)

    tail_side = rng.binomial(1, 0.5, size=n_tail)
    x_tail = rng.normal(loc=1.5, scale=0.20, size=n_tail)
    x_tail[tail_side == 0] *= -1.0
    x[~center_mask] = x_tail

    return np.clip(x, -2.0, 2.0)


def sample_x_train_test(
    n_train_base: int,
    n_cal: int,
    n_test: int,
    regime: str,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if regime == "ambiguity_only":
        x_train = sample_x_dense_uniform(n_train_base, rng)
        x_cal = sample_x_dense_uniform(n_cal, rng)
        x_test = np.linspace(-2.0, 2.0, n_test)
    elif regime == "support_imprecision":
        x_train = sample_x_dense_uniform(n_train_base, rng)
        x_cal = sample_x_center_dense_tail_sparse(n_cal, rng)
        x_test = np.linspace(-2.0, 2.0, n_test)
    else:
        raise ValueError(f"Unknown regime: {regime}")
    return x_train, x_cal, x_test


# ============================================================
# Venn-Abers wrapper
# ============================================================

@dataclass
class VAFitResult:
    p_mid: np.ndarray
    p0: np.ndarray
    p1: np.ndarray
    width: np.ndarray
    base_prob: np.ndarray


class VAOneDimBinary:
    """
    Binary base model + Venn-Abers calibration.

    This assumes a standard classifier API and a Venn-Abers calibrator
    that returns p0/p1 or equivalent interval information.

    Adjust `predict_with_interval` if your installed package returns a
    slightly different structure.
    """

    def __init__(self, random_state: int = 0):
        self.base = HistGradientBoostingClassifier(
            max_depth=4,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.va = VennAbersCalibrator(
            estimator=self.base,
            inductive=True,
            cal_size=None,
            random_state=random_state,
        )

    def fit(self, x_train: np.ndarray, y_train: np.ndarray, x_cal: np.ndarray, y_cal: np.ndarray) -> "VAOneDimBinary":
        X_train = ensure_2d_column(x_train)
        X_cal = ensure_2d_column(x_cal)

        self.base.fit(X_train, y_train)

        # Fit VA on a fixed calibration set.
        # Many implementations accept either:
        #   va.fit(X_cal, y_cal)
        # or
        #   va.fit(estimator, X_cal, y_cal)
        #
        # This line assumes the calibrator wraps the estimator and calibrates on X_cal/y_cal.
        self.va.fit(X_cal, y_cal)
        return self

    def predict_with_interval(self, x_test: np.ndarray) -> VAFitResult:
        X_test = ensure_2d_column(x_test)

        base_prob = clip01(self.base.predict_proba(X_test)[:, 1])

        # Common Venn-Abers variants return:
        #   p_prime, p0_p1
        # or
        #   p0, p1
        # This block tries to handle a couple of common patterns.
        pred, p0_p1 = self.va.predict_proba(X_test, p0_p1_output=True)
        print(p0_p1[0].shape)

        p0 = None
        p1 = None
        p_mid = None

        p0 = p0_p1[0][:,0]
        p1 = p0_p1[0][:,1]
        p_mid = pred[:,1]

        if p0 is None or p1 is None:
            raise RuntimeError(
                "Could not parse Venn-Abers interval outputs. "
                "Please adapt `predict_with_interval` to your installed library's API."
            )

        p0 = clip01(np.minimum(p0, p1))
        p1 = clip01(np.maximum(p0, p1))
        p_mid = clip01(p_mid if p_mid is not None else 0.5 * (p0 + p1))
        width = p1 - p0

        return VAFitResult(
            p_mid=p_mid,
            p0=p0,
            p1=p1,
            width=width,
            base_prob=base_prob,
        )


# ============================================================
# Diagnostics
# ============================================================

def estimate_score_density(base_probs_cal: np.ndarray, base_probs_test: np.ndarray, bandwidth: float = 0.01) -> np.ndarray:
    """
    KDE on calibration scores; returns density at test scores.
    """
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
    kde.fit(base_probs_cal.reshape(-1, 1))
    log_dens = kde.score_samples(base_probs_test.reshape(-1, 1))
    dens = np.exp(log_dens)
    return dens


def summarize_pointwise_relationships(
    x_test: np.ndarray,
    p_true: np.ndarray,
    p_mid: np.ndarray,
    width: np.ndarray,
    score_density: np.ndarray,
) -> Dict[str, float]:
    ambiguity = ambiguity_from_p(p_true)
    abs_error = np.abs(p_mid - p_true)
    inv_density = 1.0 / np.maximum(score_density, 1e-12)

    return {
        "corr_width_vs_ambiguity": corr(width, ambiguity),
        "corr_width_vs_abs_error": corr(width, abs_error),
        "corr_width_vs_inv_score_density": corr(width, inv_density),
        "mean_width": float(np.mean(width)),
        "mean_abs_error": float(np.mean(abs_error)),
        "mean_ambiguity": float(np.mean(ambiguity)),
    }


def brier_from_true_prob(p_pred: np.ndarray, p_true: np.ndarray, n_mc: int = 200, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_mc):
        y = rng.binomial(1, clip01(p_true))
        vals.append(brier_score_loss(y, clip01(p_pred)))
    return float(np.mean(vals))


# ============================================================
# Plotting
# ============================================================

def plot_main_diagnostics(
    x_test: np.ndarray,
    p_true: np.ndarray,
    va_result: VAFitResult,
    score_density: np.ndarray,
    title: str,
    filename: str,
) -> None:
    ambiguity = ambiguity_from_p(p_true)
    density_norm = score_density / np.max(score_density)
    width_norm = va_result.width / max(np.max(va_result.width), 1e-12)

    fig = plt.figure(figsize=(10, 6))
    plt.plot(x_test, p_true, label="True p(x)")
    plt.plot(x_test, va_result.p_mid, label="VA midpoint")
    plt.plot(x_test, ambiguity, label="Ambiguity 2p(1-p)")
    plt.plot(x_test, density_norm, label="Calibration score density (normalized)")
    plt.plot(x_test, width_norm, label="VA width (normalized)")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("value")
    plt.legend()
    save_plot(fig, filename)


def plot_interval_band(
    x_test: np.ndarray,
    p_true: np.ndarray,
    va_result: VAFitResult,
    title: str,
    filename: str,
) -> None:
    fig = plt.figure(figsize=(10, 5))
    plt.plot(x_test, p_true, label="True p(x)")
    plt.plot(x_test, va_result.p_mid, label="VA midpoint")
    plt.fill_between(x_test, va_result.p0, va_result.p1, alpha=0.25, label="VA interval")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("Probability")
    plt.legend()
    save_plot(fig, filename)


def plot_scatter(
    x: np.ndarray,
    y: np.ndarray,
    xlabel: str,
    ylabel: str,
    title: str,
    filename: str,
) -> None:
    fig = plt.figure(figsize=(6, 5))
    plt.scatter(x, y, s=12, alpha=0.65)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    save_plot(fig, filename)


# ============================================================
# Experiment A / B
# ============================================================

def run_single_experiment(
    regime: str,
    n_train_base: int = 3000,
    n_cal: int = 1000,
    n_test: int = 1000,
    seed: int = 0,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray], Dict[str, float]]:
    rng = np.random.default_rng(seed)

    x_train, x_cal, x_test = sample_x_train_test(
        n_train_base=n_train_base,
        n_cal=n_cal,
        n_test=n_test,
        regime=regime,
        rng=rng,
    )

    p_train = true_probability_1d(x_train)
    p_cal = true_probability_1d(x_cal)
    p_test = true_probability_1d(x_test)

    y_train = sample_labels_from_p(p_train, rng)
    y_cal = sample_labels_from_p(p_cal, rng)

    model = VAOneDimBinary(random_state=seed)
    model.fit(x_train, y_train, x_cal, y_cal)
    va_res = model.predict_with_interval(x_test)

    base_prob_cal = clip01(model.base.predict_proba(ensure_2d_column(x_cal))[:, 1])
    score_density = estimate_score_density(base_prob_cal, va_res.base_prob, bandwidth=0.04)

    summary = summarize_pointwise_relationships(
        x_test=x_test,
        p_true=p_test,
        p_mid=va_res.p_mid,
        width=va_res.width,
        score_density=score_density,
    )
    summary["brier_vs_true_distribution"] = brier_from_true_prob(va_res.p_mid, p_test, seed=seed)

    df = pd.DataFrame({
        "x_test": x_test,
        "p_true": p_test,
        "ambiguity": ambiguity_from_p(p_test),
        "va_mid": va_res.p_mid,
        "va_p0": va_res.p0,
        "va_p1": va_res.p1,
        "va_width": va_res.width,
        "base_prob": va_res.base_prob,
        "score_density": score_density,
        "inv_score_density": 1.0 / np.maximum(score_density, 1e-12),
        "abs_error": np.abs(va_res.p_mid - p_test),
    })

    artifacts = {
        "x_train": x_train,
        "x_cal": x_cal,
        "x_test": x_test,
        "p_test": p_test,
        "score_density": score_density,
        "va_width": va_res.width,
        "va_mid": va_res.p_mid,
        "ambiguity": ambiguity_from_p(p_test),
    }

    return df, artifacts, summary


# ============================================================
# Experiment C: resampling instability
# ============================================================

def run_resampling_instability_experiment(
    n_train_base: int = 3000,
    n_cal_list: List[int] = [50, 100, 200],
    n_test: int = 500,
    n_reps: int = 40,
    seed: int = 0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)

    x_train = sample_x_dense_uniform(n_train_base, rng)
    p_train = true_probability_1d(x_train)
    y_train = sample_labels_from_p(p_train, rng)

    x_test = np.linspace(-2.0, 2.0, n_test)
    p_test = true_probability_1d(x_test)

    rows_global = []
    rows_pointwise = []

    for n_cal in n_cal_list:
        mids_all = []
        widths_all = []

        for rep in range(n_reps):
            rep_rng = np.random.default_rng(seed + 1000 * n_cal + rep)
            x_cal = sample_x_dense_uniform(n_cal, rep_rng)
            p_cal = true_probability_1d(x_cal)
            y_cal = sample_labels_from_p(p_cal, rep_rng)

            model = VAOneDimBinary(random_state=seed + rep)
            model.fit(x_train, y_train, x_cal, y_cal)
            va_res = model.predict_with_interval(x_test)

            mids_all.append(va_res.p_mid)
            widths_all.append(va_res.width)

        mids_all = np.stack(mids_all, axis=0)      # [rep, n_test]
        widths_all = np.stack(widths_all, axis=0)  # [rep, n_test]

        mean_mid = mids_all.mean(axis=0)
        sd_mid = mids_all.std(axis=0)
        mean_width = widths_all.mean(axis=0)

        rows_global.append({
            "n_cal": n_cal,
            "mean_width": float(np.mean(mean_width)),
            "mean_sd_mid_across_resamples": float(np.mean(sd_mid)),
            "corr_width_vs_sd_mid": corr(mean_width, sd_mid),
            "corr_width_vs_abs_error": corr(mean_width, np.abs(mean_mid - p_test)),
            "brier_vs_true_distribution": brier_from_true_prob(mean_mid, p_test, seed=seed + n_cal),
        })

        tmp = pd.DataFrame({
            "n_cal": n_cal,
            "x_test": x_test,
            "p_true": p_test,
            "ambiguity": ambiguity_from_p(p_test),
            "mean_mid": mean_mid,
            "sd_mid_across_resamples": sd_mid,
            "mean_width": mean_width,
            "abs_error": np.abs(mean_mid - p_test),
        })
        rows_pointwise.append(tmp)

    df_global = pd.DataFrame(rows_global)
    df_pointwise = pd.concat(rows_pointwise, axis=0, ignore_index=True)
    return df_global, df_pointwise


# ============================================================
# Full run
# ============================================================

def main() -> None:
    summary_rows = []

    # ------------------------
    # Experiment A
    # ------------------------
    df_a, art_a, sum_a = run_single_experiment(
        regime="ambiguity_only",
        n_train_base=3000,
        n_cal=600,
        n_test=800,
        seed=RANDOM_SEED,
    )
    sum_a["experiment"] = "A_ambiguity_only"
    summary_rows.append(sum_a)

    plot_main_diagnostics(
        art_a["x_test"],
        art_a["p_test"],
        VAFitResult(
            p_mid=df_a["va_mid"].to_numpy(),
            p0=df_a["va_p0"].to_numpy(),
            p1=df_a["va_p1"].to_numpy(),
            width=df_a["va_width"].to_numpy(),
            base_prob=df_a["base_prob"].to_numpy(),
        ),
        art_a["score_density"],
        title="Experiment A: ambiguity only",
        filename="expA_main_diagnostics.png",
    )

    plot_interval_band(
        df_a["x_test"].to_numpy(),
        df_a["p_true"].to_numpy(),
        VAFitResult(
            p_mid=df_a["va_mid"].to_numpy(),
            p0=df_a["va_p0"].to_numpy(),
            p1=df_a["va_p1"].to_numpy(),
            width=df_a["va_width"].to_numpy(),
            base_prob=df_a["base_prob"].to_numpy(),
        ),
        title="Experiment A: VA interval band",
        filename="expA_interval_band.png",
    )

    plot_scatter(
        df_a["ambiguity"].to_numpy(),
        df_a["va_width"].to_numpy(),
        xlabel="True ambiguity",
        ylabel="VA width",
        title="Experiment A: width vs ambiguity",
        filename="expA_width_vs_ambiguity.png",
    )

    plot_scatter(
        df_a["inv_score_density"].to_numpy(),
        df_a["va_width"].to_numpy(),
        xlabel="Inverse calibration score density",
        ylabel="VA width",
        title="Experiment A: width vs inverse score density",
        filename="expA_width_vs_inv_density.png",
    )

    # ------------------------
    # Experiment B
    # ------------------------
    df_b, art_b, sum_b = run_single_experiment(
        regime="support_imprecision",
        n_train_base=3000,
        n_cal=600,
        n_test=800,
        seed=RANDOM_SEED + 1,
    )
    sum_b["experiment"] = "B_support_imprecision"
    summary_rows.append(sum_b)

    plot_main_diagnostics(
        art_b["x_test"],
        art_b["p_test"],
        VAFitResult(
            p_mid=df_b["va_mid"].to_numpy(),
            p0=df_b["va_p0"].to_numpy(),
            p1=df_b["va_p1"].to_numpy(),
            width=df_b["va_width"].to_numpy(),
            base_prob=df_b["base_prob"].to_numpy(),
        ),
        art_b["score_density"],
        title="Experiment B: support imprecision",
        filename="expB_main_diagnostics.png",
    )

    plot_interval_band(
        df_b["x_test"].to_numpy(),
        df_b["p_true"].to_numpy(),
        VAFitResult(
            p_mid=df_b["va_mid"].to_numpy(),
            p0=df_b["va_p0"].to_numpy(),
            p1=df_b["va_p1"].to_numpy(),
            width=df_b["va_width"].to_numpy(),
            base_prob=df_b["base_prob"].to_numpy(),
        ),
        title="Experiment B: VA interval band",
        filename="expB_interval_band.png",
    )

    plot_scatter(
        df_b["ambiguity"].to_numpy(),
        df_b["va_width"].to_numpy(),
        xlabel="True ambiguity",
        ylabel="VA width",
        title="Experiment B: width vs ambiguity",
        filename="expB_width_vs_ambiguity.png",
    )

    plot_scatter(
        df_b["inv_score_density"].to_numpy(),
        df_b["va_width"].to_numpy(),
        xlabel="Inverse calibration score density",
        ylabel="VA width",
        title="Experiment B: width vs inverse score density",
        filename="expB_width_vs_inv_density.png",
    )

    # ------------------------
    # Experiment C
    # ------------------------
    df_c_global, df_c_pointwise = run_resampling_instability_experiment(
        n_train_base=3000,
        n_cal_list=[50, 100, 250, 500, 1000, 2000],
        n_test=600,
        n_reps=40,
        seed=RANDOM_SEED + 2,
    )

    fig = plt.figure(figsize=(7, 5))
    plt.plot(df_c_global["n_cal"], df_c_global["mean_width"], marker="o", label="Mean VA width")
    plt.plot(df_c_global["n_cal"], df_c_global["mean_sd_mid_across_resamples"], marker="o", label="Mean SD of midpoint across resamples")
    plt.xscale("log")
    plt.xlabel("Calibration size")
    plt.ylabel("Value")
    plt.title("Experiment C: shrinking width and instability with larger calibration sets")
    plt.legend()
    save_plot(fig, "expC_width_and_instability_vs_ncal.png")

    # For one representative calibration size, show width vs resampling SD
    rep_n_cal = 100
    df_rep = df_c_pointwise[df_c_pointwise["n_cal"] == rep_n_cal].copy()
    plot_scatter(
        df_rep["sd_mid_across_resamples"].to_numpy(),
        df_rep["mean_width"].to_numpy(),
        xlabel="SD of midpoint across calibration resamples",
        ylabel="Mean VA width",
        title=f"Experiment C: width vs resampling instability (n_cal={rep_n_cal})",
        filename="expC_width_vs_resampling_sd.png",
    )

    for _, row in df_c_global.iterrows():
        summary_rows.append({
            "experiment": f"C_resampling_ncal_{int(row['n_cal'])}",
            "corr_width_vs_ambiguity": np.nan,
            "corr_width_vs_abs_error": row["corr_width_vs_abs_error"],
            "corr_width_vs_inv_score_density": np.nan,
            "mean_width": row["mean_width"],
            "mean_abs_error": np.nan,
            "mean_ambiguity": np.nan,
            "brier_vs_true_distribution": row["brier_vs_true_distribution"],
            "corr_width_vs_sd_mid": row["corr_width_vs_sd_mid"],
            "mean_sd_mid_across_resamples": row["mean_sd_mid_across_resamples"],
        })

    # Save tables
    df_a.to_csv(os.path.join(OUTPUT_DIR, "experiment_A_pointwise.csv"), index=False)
    df_b.to_csv(os.path.join(OUTPUT_DIR, "experiment_B_pointwise.csv"), index=False)
    df_c_global.to_csv(os.path.join(OUTPUT_DIR, "experiment_C_global.csv"), index=False)
    df_c_pointwise.to_csv(os.path.join(OUTPUT_DIR, "experiment_C_pointwise.csv"), index=False)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)

    print("\n=== Summary table ===")
    print(summary_df.round(4))

    print(f"\nSaved outputs to: {OUTPUT_DIR}")

import warnings
warnings.filterwarnings("ignore")

In [21]:
main()

(800, 2)
(800, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(600, 2)
(

In [23]:
from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, List, Tuple

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.neighbors import NearestNeighbors

from venn_abers import VennAbersCalibrator


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 13
OUTPUT_DIR = "va_calibration_imprecision_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# Basic utilities
# ============================================================

def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def clip01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return np.clip(x, eps, 1.0 - eps)


def ensure_2d_column(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    if x.ndim == 1:
        return x.reshape(-1, 1)
    return x


def corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def save_plot(fig: plt.Figure, filename: str) -> None:
    path = os.path.join(OUTPUT_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Synthetic truth
# ============================================================

def true_probability_1d(x: np.ndarray) -> np.ndarray:
    """
    Smooth monotone truth.
    """
    return 0.1 + 0.8 * sigmoid(6.0 * x)


def ambiguity_from_p(p: np.ndarray) -> np.ndarray:
    return 2.0 * p * (1.0 - p)


def sample_labels_from_p(p: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return rng.binomial(1, clip01(p)).astype(int)


# ============================================================
# Sampling regimes
# ============================================================

def sample_x_dense_uniform(n: int, rng: np.random.Generator) -> np.ndarray:
    return rng.uniform(-2.0, 2.0, size=n)


def sample_x_center_dense_tail_sparse(n: int, rng: np.random.Generator) -> np.ndarray:
    """
    Very dense near 0, very sparse near the extremes.
    This makes support differences more pronounced.
    """
    mix = rng.uniform(size=n)
    x = np.empty(n)

    center_mask = mix < 0.92
    n_center = center_mask.sum()
    n_tail = n - n_center

    x[center_mask] = rng.normal(loc=0.0, scale=0.20, size=n_center)

    tail_side = rng.binomial(1, 0.5, size=n_tail)
    x_tail = rng.normal(loc=1.75, scale=0.08, size=n_tail)
    x_tail[tail_side == 0] *= -1.0
    x[~center_mask] = x_tail

    return np.clip(x, -2.0, 2.0)


def sample_x_train_test(
    n_train_base: int,
    n_cal: int,
    n_test: int,
    regime: str,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if regime == "ambiguity_only":
        x_train = sample_x_dense_uniform(n_train_base, rng)
        x_cal = sample_x_dense_uniform(n_cal, rng)
        x_test = np.linspace(-2.0, 2.0, n_test)
    elif regime == "support_imprecision":
        x_train = sample_x_dense_uniform(n_train_base, rng)
        x_cal = sample_x_center_dense_tail_sparse(n_cal, rng)
        x_test = np.linspace(-2.0, 2.0, n_test)
    else:
        raise ValueError(f"Unknown regime: {regime}")

    return x_train, x_cal, x_test


# ============================================================
# Venn-Abers wrapper
# ============================================================

@dataclass
class VAFitResult:
    p_mid: np.ndarray
    p0: np.ndarray
    p1: np.ndarray
    width: np.ndarray
    base_prob: np.ndarray


class VAOneDimBinary:
    """
    Binary classifier + Venn-Abers calibration wrapper.

    This uses:
      predict_proba(..., p0_p1_output=True)
    to obtain true p0 / p1 bounds from the library.
    """

    def __init__(self, random_state: int = 0):
        self.base = HistGradientBoostingClassifier(
            max_depth=4,
            learning_rate=0.05,
            max_iter=250,
            min_samples_leaf=20,
            random_state=random_state,
        )
        self.va = VennAbersCalibrator(
            estimator=self.base,
            inductive=True,
            cal_size=None,
            random_state=random_state,
        )

    def fit(
        self,
        x_train: np.ndarray,
        y_train: np.ndarray,
        x_cal: np.ndarray,
        y_cal: np.ndarray,
    ) -> "VAOneDimBinary":
        X_train = ensure_2d_column(x_train)
        X_cal = ensure_2d_column(x_cal)

        self.base.fit(X_train, y_train)
        self.va.fit(X_cal, y_cal)
        return self

    def predict_with_interval(self, x_test: np.ndarray) -> VAFitResult:
        X_test = ensure_2d_column(x_test)

        base_prob = clip01(self.base.predict_proba(X_test)[:, 1])

        pred, p0_p1 = self.va.predict_proba(X_test, p0_p1_output=True)
        # print(p0_p1[0].shape)

        p0 = None
        p1 = None
        p_mid = None

        p0 = p0_p1[0][:,0]
        p1 = p0_p1[0][:,1]
        p_mid = pred[:,1]

        if p0 is None or p1 is None:
            raise RuntimeError(
                "Could not parse Venn-Abers p0/p1 output. "
                "Inspect `self.va.predict_proba(X_test, p0_p1_output=True)` "
                "in your environment and adapt the parser if needed."
            )

        p0_raw = np.asarray(p0, dtype=float).reshape(-1)
        p1_raw = np.asarray(p1, dtype=float).reshape(-1)

        p0_final = clip01(np.minimum(p0_raw, p1_raw))
        p1_final = clip01(np.maximum(p0_raw, p1_raw))

        if p_mid is None:
            p_mid_final = 0.5 * (p0_final + p1_final)
        else:
            p_mid_final = clip01(np.asarray(p_mid, dtype=float).reshape(-1))

        return VAFitResult(
            p_mid=p_mid_final,
            p0=p0_final,
            p1=p1_final,
            width=p1_final - p0_final,
            base_prob=base_prob,
        )


# ============================================================
# Local support metrics in score space
# ============================================================

def local_support_eps(
    cal_scores: np.ndarray,
    test_scores: np.ndarray,
    eps: float = 0.02,
) -> np.ndarray:
    """
    Count how many calibration scores lie within eps of each test score.
    Larger = stronger local support.
    """
    cal_scores = np.asarray(cal_scores, dtype=float).reshape(-1, 1)
    test_scores = np.asarray(test_scores, dtype=float).reshape(-1, 1)

    nn = NearestNeighbors(radius=eps, metric="euclidean")
    nn.fit(cal_scores)
    neigh_ind = nn.radius_neighbors(test_scores, return_distance=False)
    counts = np.array([len(ix) for ix in neigh_ind], dtype=float)
    return counts


def local_support_knn_distance(
    cal_scores: np.ndarray,
    test_scores: np.ndarray,
    k: int = 15,
) -> np.ndarray:
    """
    Distance to the k-th nearest calibration score.
    Larger = weaker local support.
    """
    cal_scores = np.asarray(cal_scores, dtype=float).reshape(-1, 1)
    test_scores = np.asarray(test_scores, dtype=float).reshape(-1, 1)

    k = min(k, len(cal_scores))
    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(cal_scores)
    dists, _ = nn.kneighbors(test_scores)
    kth_dist = dists[:, -1]
    return kth_dist


# ============================================================
# Diagnostics and summaries
# ============================================================

def brier_from_true_prob(
    p_pred: np.ndarray,
    p_true: np.ndarray,
    n_mc: int = 200,
    seed: int = 0,
) -> float:
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_mc):
        y = rng.binomial(1, clip01(p_true))
        vals.append(brier_score_loss(y, clip01(p_pred)))
    return float(np.mean(vals))


def summarize_pointwise_relationships(
    p_true: np.ndarray,
    p_mid: np.ndarray,
    width: np.ndarray,
    knn_dist: np.ndarray,
    eps_counts: np.ndarray,
) -> Dict[str, float]:
    ambiguity = ambiguity_from_p(p_true)
    abs_error = np.abs(p_mid - p_true)

    return {
        "corr_width_vs_ambiguity": corr(width, ambiguity),
        "corr_width_vs_abs_error": corr(width, abs_error),
        "corr_width_vs_knn_dist": corr(width, knn_dist),
        "corr_width_vs_inv_eps_count": corr(width, 1.0 / (eps_counts + 1.0)),
        "mean_width": float(np.mean(width)),
        "mean_abs_error": float(np.mean(abs_error)),
        "mean_ambiguity": float(np.mean(ambiguity)),
        "mean_knn_dist": float(np.mean(knn_dist)),
        "mean_eps_count": float(np.mean(eps_counts)),
    }


# ============================================================
# Plotting
# ============================================================

def plot_main_diagnostics(
    x_test: np.ndarray,
    p_true: np.ndarray,
    va_result: VAFitResult,
    knn_dist: np.ndarray,
    eps_counts: np.ndarray,
    title: str,
    filename: str,
) -> None:
    ambiguity = ambiguity_from_p(p_true)
    width_norm = va_result.width / max(np.max(va_result.width), 1e-12)
    knn_norm = knn_dist / max(np.max(knn_dist), 1e-12)
    inv_eps = 1.0 / (eps_counts + 1.0)
    inv_eps_norm = inv_eps / max(np.max(inv_eps), 1e-12)

    fig = plt.figure(figsize=(10, 6))
    plt.plot(x_test, p_true, label="True p(x)")
    plt.plot(x_test, va_result.p_mid, label="VA midpoint")
    plt.plot(x_test, ambiguity, label="Ambiguity 2p(1-p)")
    plt.plot(x_test, width_norm, label="VA width (normalized)")
    plt.plot(x_test, knn_norm, label="kNN score distance (normalized)")
    plt.plot(x_test, inv_eps_norm, label="Inverse eps-count (normalized)")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("value")
    plt.legend()
    save_plot(fig, filename)


def plot_interval_band(
    x_test: np.ndarray,
    p_true: np.ndarray,
    va_result: VAFitResult,
    title: str,
    filename: str,
) -> None:
    fig = plt.figure(figsize=(10, 5))
    plt.plot(x_test, p_true, label="True p(x)")
    plt.plot(x_test, va_result.p_mid, label="VA midpoint")
    plt.fill_between(x_test, va_result.p0, va_result.p1, alpha=0.25, label="VA interval")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("probability")
    plt.legend()
    save_plot(fig, filename)


def plot_scatter(
    x: np.ndarray,
    y: np.ndarray,
    xlabel: str,
    ylabel: str,
    title: str,
    filename: str,
) -> None:
    fig = plt.figure(figsize=(6, 5))
    plt.scatter(x, y, s=14, alpha=0.65)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    save_plot(fig, filename)


def plot_calibration_histogram(
    cal_scores: np.ndarray,
    title: str,
    filename: str,
) -> None:
    fig = plt.figure(figsize=(7, 4))
    plt.hist(cal_scores, bins=40, alpha=0.8)
    plt.xlabel("Base-model calibration scores")
    plt.ylabel("Count")
    plt.title(title)
    save_plot(fig, filename)


# ============================================================
# Experiment A / B
# ============================================================

def run_single_experiment(
    regime: str,
    n_train_base: int = 3000,
    n_cal: int = 600,
    n_test: int = 800,
    seed: int = 0,
) -> Tuple[pd.DataFrame, Dict[str, float], Dict[str, np.ndarray]]:
    rng = np.random.default_rng(seed)

    x_train, x_cal, x_test = sample_x_train_test(
        n_train_base=n_train_base,
        n_cal=n_cal,
        n_test=n_test,
        regime=regime,
        rng=rng,
    )

    p_train = true_probability_1d(x_train)
    p_cal = true_probability_1d(x_cal)
    p_test = true_probability_1d(x_test)

    y_train = sample_labels_from_p(p_train, rng)
    y_cal = sample_labels_from_p(p_cal, rng)

    model = VAOneDimBinary(random_state=seed)
    model.fit(x_train, y_train, x_cal, y_cal)
    va_res = model.predict_with_interval(x_test)

    base_prob_cal = clip01(model.base.predict_proba(ensure_2d_column(x_cal))[:, 1])

    knn_dist = local_support_knn_distance(
        cal_scores=base_prob_cal,
        test_scores=va_res.base_prob,
        k=15,
    )
    eps_counts = local_support_eps(
        cal_scores=base_prob_cal,
        test_scores=va_res.base_prob,
        eps=0.02,
    )

    summary = summarize_pointwise_relationships(
        p_true=p_test,
        p_mid=va_res.p_mid,
        width=va_res.width,
        knn_dist=knn_dist,
        eps_counts=eps_counts,
    )
    summary["brier_vs_true_distribution"] = brier_from_true_prob(
        va_res.p_mid, p_test, seed=seed
    )

    df = pd.DataFrame({
        "x_test": x_test,
        "p_true": p_test,
        "ambiguity": ambiguity_from_p(p_test),
        "va_mid": va_res.p_mid,
        "va_p0": va_res.p0,
        "va_p1": va_res.p1,
        "va_width": va_res.width,
        "base_prob": va_res.base_prob,
        "knn_dist": knn_dist,
        "eps_counts": eps_counts,
        "inv_eps_count": 1.0 / (eps_counts + 1.0),
        "abs_error": np.abs(va_res.p_mid - p_test),
    })

    artifacts = {
        "x_train": x_train,
        "x_cal": x_cal,
        "x_test": x_test,
        "p_test": p_test,
        "base_prob_cal": base_prob_cal,
        "knn_dist": knn_dist,
        "eps_counts": eps_counts,
        "va_mid": va_res.p_mid,
        "va_p0": va_res.p0,
        "va_p1": va_res.p1,
        "va_width": va_res.width,
        "base_prob_test": va_res.base_prob,
    }

    return df, summary, artifacts


# ============================================================
# Experiment C: calibration resampling instability
# ============================================================

def run_resampling_instability_experiment(
    n_train_base: int = 3000,
    n_cal_list: List[int] = [50, 100, 250, 500, 1000, 2000],
    n_test: int = 600,
    n_reps: int = 40,
    seed: int = 0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)

    x_train = sample_x_dense_uniform(n_train_base, rng)
    p_train = true_probability_1d(x_train)
    y_train = sample_labels_from_p(p_train, rng)

    x_test = np.linspace(-2.0, 2.0, n_test)
    p_test = true_probability_1d(x_test)

    rows_global = []
    rows_pointwise = []

    for n_cal in n_cal_list:
        mids_all = []
        widths_all = []

        for rep in range(n_reps):
            rep_rng = np.random.default_rng(seed + 1000 * n_cal + rep)
            x_cal = sample_x_dense_uniform(n_cal, rep_rng)
            p_cal = true_probability_1d(x_cal)
            y_cal = sample_labels_from_p(p_cal, rep_rng)

            model = VAOneDimBinary(random_state=seed + rep)
            model.fit(x_train, y_train, x_cal, y_cal)
            va_res = model.predict_with_interval(x_test)

            mids_all.append(va_res.p_mid)
            widths_all.append(va_res.width)

        mids_all = np.stack(mids_all, axis=0)
        widths_all = np.stack(widths_all, axis=0)

        mean_mid = mids_all.mean(axis=0)
        sd_mid = mids_all.std(axis=0)
        mean_width = widths_all.mean(axis=0)
        abs_error = np.abs(mean_mid - p_test)

        rows_global.append({
            "n_cal": n_cal,
            "mean_width": float(np.mean(mean_width)),
            "mean_sd_mid_across_resamples": float(np.mean(sd_mid)),
            "corr_width_vs_sd_mid": corr(mean_width, sd_mid),
            "corr_width_vs_abs_error": corr(mean_width, abs_error),
            "brier_vs_true_distribution": brier_from_true_prob(mean_mid, p_test, seed=seed + n_cal),
        })

        rows_pointwise.append(pd.DataFrame({
            "n_cal": n_cal,
            "x_test": x_test,
            "p_true": p_test,
            "ambiguity": ambiguity_from_p(p_test),
            "mean_mid": mean_mid,
            "sd_mid_across_resamples": sd_mid,
            "mean_width": mean_width,
            "abs_error": abs_error,
        }))

    df_global = pd.DataFrame(rows_global)
    df_pointwise = pd.concat(rows_pointwise, axis=0, ignore_index=True)
    return df_global, df_pointwise


# ============================================================
# Main
# ============================================================

def main() -> None:
    summary_rows = []

    # ------------------------
    # Experiment A
    # ------------------------
    df_a, sum_a, art_a = run_single_experiment(
        regime="ambiguity_only",
        n_train_base=3000,
        n_cal=600,
        n_test=800,
        seed=RANDOM_SEED,
    )
    sum_a["experiment"] = "A_ambiguity_only"
    summary_rows.append(sum_a)

    va_a = VAFitResult(
        p_mid=art_a["va_mid"],
        p0=art_a["va_p0"],
        p1=art_a["va_p1"],
        width=art_a["va_width"],
        base_prob=art_a["base_prob_test"],
    )

    plot_main_diagnostics(
        art_a["x_test"],
        art_a["p_test"],
        va_a,
        art_a["knn_dist"],
        art_a["eps_counts"],
        title="Experiment A: ambiguity only",
        filename="expA_main_diagnostics.png",
    )

    plot_interval_band(
        art_a["x_test"],
        art_a["p_test"],
        va_a,
        title="Experiment A: VA interval band",
        filename="expA_interval_band.png",
    )

    plot_scatter(
        df_a["ambiguity"].to_numpy(),
        df_a["va_width"].to_numpy(),
        xlabel="True ambiguity",
        ylabel="VA width",
        title="Experiment A: width vs ambiguity",
        filename="expA_width_vs_ambiguity.png",
    )

    plot_scatter(
        df_a["knn_dist"].to_numpy(),
        df_a["va_width"].to_numpy(),
        xlabel="kNN distance in calibration-score space",
        ylabel="VA width",
        title="Experiment A: width vs local sparsity",
        filename="expA_width_vs_knn_dist.png",
    )

    plot_scatter(
        df_a["inv_eps_count"].to_numpy(),
        df_a["va_width"].to_numpy(),
        xlabel="Inverse epsilon-neighborhood count",
        ylabel="VA width",
        title="Experiment A: width vs inverse local support",
        filename="expA_width_vs_inv_eps_count.png",
    )

    plot_calibration_histogram(
        art_a["base_prob_cal"],
        title="Experiment A: calibration score histogram",
        filename="expA_calibration_score_hist.png",
    )

    # ------------------------
    # Experiment B
    # ------------------------
    df_b, sum_b, art_b = run_single_experiment(
        regime="support_imprecision",
        n_train_base=3000,
        n_cal=600,
        n_test=800,
        seed=RANDOM_SEED + 1,
    )
    sum_b["experiment"] = "B_support_imprecision"
    summary_rows.append(sum_b)

    va_b = VAFitResult(
        p_mid=art_b["va_mid"],
        p0=art_b["va_p0"],
        p1=art_b["va_p1"],
        width=art_b["va_width"],
        base_prob=art_b["base_prob_test"],
    )

    plot_main_diagnostics(
        art_b["x_test"],
        art_b["p_test"],
        va_b,
        art_b["knn_dist"],
        art_b["eps_counts"],
        title="Experiment B: support imprecision",
        filename="expB_main_diagnostics.png",
    )

    plot_interval_band(
        art_b["x_test"],
        art_b["p_test"],
        va_b,
        title="Experiment B: VA interval band",
        filename="expB_interval_band.png",
    )

    plot_scatter(
        df_b["ambiguity"].to_numpy(),
        df_b["va_width"].to_numpy(),
        xlabel="True ambiguity",
        ylabel="VA width",
        title="Experiment B: width vs ambiguity",
        filename="expB_width_vs_ambiguity.png",
    )

    plot_scatter(
        df_b["knn_dist"].to_numpy(),
        df_b["va_width"].to_numpy(),
        xlabel="kNN distance in calibration-score space",
        ylabel="VA width",
        title="Experiment B: width vs local score-space sparsity",
        filename="expB_width_vs_knn_dist.png",
    )

    plot_scatter(
        df_b["inv_eps_count"].to_numpy(),
        df_b["va_width"].to_numpy(),
        xlabel="Inverse epsilon-neighborhood count",
        ylabel="VA width",
        title="Experiment B: width vs inverse local support",
        filename="expB_width_vs_inv_eps_count.png",
    )

    plot_calibration_histogram(
        art_b["base_prob_cal"],
        title="Experiment B: calibration score histogram",
        filename="expB_calibration_score_hist.png",
    )

    # ------------------------
    # Experiment C
    # ------------------------
    df_c_global, df_c_pointwise = run_resampling_instability_experiment(
        n_train_base=3000,
        n_cal_list=[50, 100, 250, 500, 1000, 2000],
        n_test=600,
        n_reps=40,
        seed=RANDOM_SEED + 2,
    )

    fig = plt.figure(figsize=(7, 5))
    plt.plot(df_c_global["n_cal"], df_c_global["mean_width"], marker="o", label="Mean VA width")
    plt.plot(df_c_global["n_cal"], df_c_global["mean_sd_mid_across_resamples"], marker="o",
             label="Mean SD of midpoint across resamples")
    plt.xscale("log")
    plt.xlabel("Calibration size")
    plt.ylabel("value")
    plt.title("Experiment C: width and instability vs calibration size")
    plt.legend()
    save_plot(fig, "expC_width_and_instability_vs_ncal.png")

    rep_n_cal = 100
    df_rep = df_c_pointwise[df_c_pointwise["n_cal"] == rep_n_cal].copy()

    plot_scatter(
        df_rep["sd_mid_across_resamples"].to_numpy(),
        df_rep["mean_width"].to_numpy(),
        xlabel="SD of midpoint across calibration resamples",
        ylabel="Mean VA width",
        title=f"Experiment C: width vs resampling instability (n_cal={rep_n_cal})",
        filename="expC_width_vs_resampling_sd.png",
    )

    for _, row in df_c_global.iterrows():
        summary_rows.append({
            "experiment": f"C_resampling_ncal_{int(row['n_cal'])}",
            "corr_width_vs_ambiguity": np.nan,
            "corr_width_vs_abs_error": row["corr_width_vs_abs_error"],
            "corr_width_vs_knn_dist": np.nan,
            "corr_width_vs_inv_eps_count": np.nan,
            "mean_width": row["mean_width"],
            "mean_abs_error": np.nan,
            "mean_ambiguity": np.nan,
            "mean_knn_dist": np.nan,
            "mean_eps_count": np.nan,
            "brier_vs_true_distribution": row["brier_vs_true_distribution"],
            "corr_width_vs_sd_mid": row["corr_width_vs_sd_mid"],
            "mean_sd_mid_across_resamples": row["mean_sd_mid_across_resamples"],
        })

    # ------------------------
    # Save outputs
    # ------------------------
    df_a.to_csv(os.path.join(OUTPUT_DIR, "experiment_A_pointwise.csv"), index=False)
    df_b.to_csv(os.path.join(OUTPUT_DIR, "experiment_B_pointwise.csv"), index=False)
    df_c_global.to_csv(os.path.join(OUTPUT_DIR, "experiment_C_global.csv"), index=False)
    df_c_pointwise.to_csv(os.path.join(OUTPUT_DIR, "experiment_C_pointwise.csv"), index=False)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_table.csv"), index=False)

    print("\n=== Summary table ===")
    print(summary_df.round(4))
    print(f"\nSaved outputs to: {OUTPUT_DIR}")



In [24]:
main()


=== Summary table ===
   corr_width_vs_ambiguity  corr_width_vs_abs_error  corr_width_vs_knn_dist  \
0                   0.3318                   0.4089                  0.2480   
1                  -0.4361                  -0.5862                  0.2575   
2                      NaN                      NaN                     NaN   
3                      NaN                  -0.1777                     NaN   
4                      NaN                  -0.2456                     NaN   
5                      NaN                  -0.2436                     NaN   
6                      NaN                  -0.4283                     NaN   
7                      NaN                   0.0124                     NaN   

   corr_width_vs_inv_eps_count  mean_width  mean_abs_error  mean_ambiguity  \
0                       0.2357      0.0432           0.072          0.2333   
1                       0.1115      0.0718           0.058          0.2333   
2                          NaN 